In [1]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# 1. Setup Environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["TRAINING_ENV"] = "kaggle"

# 2. Clone Repository
REPO_URL = "https://github.com/Firojpaudel/chatterbox-nepali.git"
REPO_DIR = "/kaggle/working/chatterbox-nepali"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# 3. Install Dependencies
print("Installing dependencies...")
# 💡 FIX: Explicitly upgrade numpy/pandas/datasets first to avoid binary incompatibility errors
subprocess.run(["pip", "install", "--upgrade", "numpy", "pandas", "datasets"], check=True)
subprocess.run(["pip", "install", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run(["pip", "install", "-e", "."], check=True)
subprocess.run(["pip", "install", "omegaconf", "conformer", "s3tokenizer", "safetensors", "pyloudnorm", "wandb", "huggingface-hub", "librosa", "soundfile"], check=True)
subprocess.run(["pip", "install", "git+https://github.com/resemble-ai/Perth.git@master"], check=True)

print("\n--- Chatterbox Setup Complete ---")

In [2]:
%cd /kaggle/working/chatterbox-nepali

In [3]:
# Download existing checkpoint from Hugging Face
from huggingface_hub import hf_hub_download
print("Downloading t3_nepali_epoch_20.pt...")
ckpt_path = hf_hub_download(repo_id="officialuser/chatterbox-nepali", filename="t3_nepali_epoch_20.pt")
dest = "t3_nepali_epoch_20.pt"
if os.path.exists(dest): os.remove(dest)
os.symlink(ckpt_path, dest)
print(f"✅ Checkpoint ready at {dest}")

In [5]:
# Download and Extract Dataset
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download

dataset_repo = "Firoj112/voxcpm-nepali-data"
local_dir = Path("finetuning_data")
wavs_dir = local_dir / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

print(f"Pulling manifests from {dataset_repo}...")
snapshot_download(
    repo_id=dataset_repo, 
    repo_type="dataset", 
    local_dir=local_dir,
    allow_patterns=["manifests/*.jsonl"],
    token=os.environ.get("HF_TOKEN")
)

try:
    from datasets import load_dataset
    import soundfile as sf
    from tqdm.auto import tqdm

    print(f"Extracting audio from Parquet files...")
    ds = load_dataset(dataset_repo, token=os.environ.get("HF_TOKEN"))

    for split in ds.keys():
        print(f"Processing split: {split}")
        for item in tqdm(ds[split]):
            audio = item['audio']
            if 'path' in audio and audio['path']:
                filename = Path(audio['path']).name
            else:
                filename = f"{item.get('id', 'unknown')}.wav"
            
            save_path = wavs_dir / filename
            if not save_path.exists():
                sf.write(save_path, audio['array'], audio['sampling_rate'])
    print(f"✅ Dataset ready: {len(list(wavs_dir.glob('*.wav')))} WAV files extracted.")
except ImportError as e:
    print(f"❌ Dependency error: {e}. Please restart the Kaggle session and run Cell 1 again.")
except Exception as e:
    print(f"❌ Failed to extract audio: {e}")

In [7]:
# 🚀 START DISTRIBUTED TRAINING (T4 x 2)
!torchrun --nproc_per_node=2 src/chatterbox/train_nepali.py \
    --manifest finetuning_data/manifests/train_manifest.jsonl \
    --resume_t3_weights t3_nepali_epoch_20.pt \
    --batch_size 16 \
    --epochs 50 \
    --lr 2e-5 \
    --save_every 5 \
    --num_workers 2 \
    --distributed